In [5]:
#Importing all the necessary libraries
import pandas as pd
import numpy as np

In [6]:
#Loading data
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")

#Filter for delivered orders to exclude cancellations
orders = orders[orders["order_status"] == "delivered"].copy()
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

#Aggregate item price and freight per order
order_rev = (
    order_items.groupby("order_id")
    .agg(
        item_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
    )
    .reset_index()
)

#Merge orders with items and customer unique IDs
df = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="inner",
)
df = df.merge(order_rev, on="order_id", how="inner")

In [7]:
# 2. FINANCIAL DERIVATION (REVENUE, COGS, GROSS MARGIN)
# ---------------------------------------------------------
# Total GMV = Item Price + Freight
df["total_gmv"] = df["item_revenue"] + df["freight_value"]

# In an e-commerce marketplace:
# Marketplace take-rate / commission: 16% of GMV
# Payment processing + variable operational costs: 4% of GMV
# Net Revenue = 16% of GMV; COGS = 4% of GMV -> Gross Margin = 12% of GMV
df["net_revenue"] = df["total_gmv"] * 0.16
df["cogs"] = df["total_gmv"] * 0.04
df["gross_margin"] = df["net_revenue"] - df["cogs"]

In [8]:
# 3. COHORT ASSIGNMENT & RETENTION
# ---------------------------------------------------------
# Monthly time buckets
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")

# Assign each customer's first purchase month (Cohort Month)
df["cohort_month"] = df.groupby("customer_unique_id")[
    "order_month"
].transform("min")

# Filter out early 2016 test data & incomplete final months (Focus on Jan 2017 – Aug 2018)
df = df[
    (df["cohort_month"] >= "2017-01") & (df["cohort_month"] <= "2018-08")
].copy()

# Compute Cohort Index (Months elapsed since acquisition)
df["cohort_index"] = (
    (df["order_month"].dt.year - df["cohort_month"].dt.year) * 12
    + (df["order_month"].dt.month - df["cohort_month"].dt.month)
)

# Customer count matrix
cohort_counts = (
    df.groupby(["cohort_month", "cohort_index"])["customer_unique_id"]
    .nunique()
    .reset_index()
)

cohort_matrix = cohort_counts.pivot(
    index="cohort_month", columns="cohort_index", values="customer_unique_id"
)

cohort_sizes = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0) * 100

In [9]:
# 4. CUMULATIVE GROSS MARGIN & EMPIRICAL LTV
# ---------------------------------------------------------
# Total gross margin per cohort per elapsed month
cohort_margins = (
    df.groupby(["cohort_month", "cohort_index"])["gross_margin"]
    .sum()
    .reset_index()
)

margin_matrix = cohort_margins.pivot(
    index="cohort_month", columns="cohort_index", values="gross_margin"
).fillna(0)

# Cumulative Gross Margin
cumulative_margin = margin_matrix.cumsum(axis=1)

# LTV trajectory per customer (R$ per acquired user)
ltv_matrix = cumulative_margin.divide(cohort_sizes, axis=0)

In [10]:
# 5. CAC SIMULATION & PAYBACK PERIOD
# ---------------------------------------------------------
# Market acquisition spend simulation:
# Base fixed paid acquisition: R$ 25,000/month + R$ 18 per acquired user
monthly_marketing_spend = pd.Series(
    25000 + (cohort_sizes * 18), index=cohort_sizes.index
)

# Blended CAC per acquired customer
blended_cac = monthly_marketing_spend / cohort_sizes

# Calculate Payback Month (First month where LTV >= CAC)
payback_results = {}
for cohort in ltv_matrix.index:
    cac_target = blended_cac.loc[cohort]
    recovered = ltv_matrix.loc[cohort] >= cac_target
    payback_results[cohort] = (
        int(recovered.idxmax()) if recovered.any() else np.nan
    )

unit_economics = pd.DataFrame(
    {
        "Acquired_Customers": cohort_sizes,
        "Simulated_Ad_Spend": monthly_marketing_spend.round(2),
        "Blended_CAC": blended_cac.round(2),
        "Initial_M0_LTV": ltv_matrix[0].round(2),
        "6M_Cumulative_LTV": ltv_matrix[5].round(2)
        if 5 in ltv_matrix.columns
        else np.nan,
        "12M_Cumulative_LTV": ltv_matrix[11].round(2)
        if 11 in ltv_matrix.columns
        else np.nan,
        "Payback_Period_Months": pd.Series(payback_results),
    }
)

unit_economics["LTV_to_CAC_12M"] = (
    unit_economics["12M_Cumulative_LTV"] / unit_economics["Blended_CAC"]
).round(2)

In [ ]:
# 6. EXPORT CLEAN TABLES FOR DASHBOARD IMPORT
# ---------------------------------------------------------
# Normalized transaction-level table with customer cohorts
df[
    [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "cohort_month",
        "cohort_index",
        "total_gmv",
        "net_revenue",
        "gross_margin",
    ]
].to_csv("1_fact_table.csv", index=False)

retention_matrix.round(2).to_csv("olist_retention_matrix.csv")
ltv_matrix.round(2).to_csv("olist_cumulative_ltv.csv")
unit_economics.to_csv("2_dim_table.csv")